# Phase 6e Wave 1: Seeley-DeWitt Heat-Kernel Expansion — Technical Notebook

Companion to `papers/paper39_heat_kernel_expansion/paper_draft.tex`.

**Lean module:** `lean/SKEFTHawking/HeatKernelExpansion.lean` (19 substantive theorems, 0 sorry, 0 new axioms; verified `propext, Classical.choice, Quot.sound` only via `lean_verify`).

**Structure (mirrors paper §1–§6):**
1. Setup and Christensen-Duff coefficients (Vassilevich 2003 + Gilkey 1995)
2. Closed-form $a_0$ leading coefficient
3. Closed-form $a_2$ Einstein-Hilbert coefficient
4. Tracked-hypothesis structure `DiracHeatKernelAsymptotic`
5. Decision Gate E.2: $G_N$ from $a_2$ ↔ Sakharov-Adler $G_N$
6. Decision Gate E.2 biconditional vs $\alpha_{\mathrm{ADW}}$
7. Higher-curvature $a_4$ basis + Gauss-Bonnet local algebra
8. Figure: calibration scan + $G_N(\Lambda_{UV})$ across natural range
9. Lean theorem inventory (full)

All numerical content imports from `src.heat_kernel` — no inline physics redefinition.


## 1. Setup and Christensen-Duff coefficients

The Seeley-DeWitt asymptotic expansion of the Dirac heat kernel,

$$\mathrm{Tr}\,e^{-\tau\,\mathit{D}\!\!\!\!/^{\,2}} \;\sim\; \frac{1}{(4\pi\tau)^2}\sum_{n \ge 0} \tau^n \int d^4 x \sqrt{g}\,a_{2n}(x), \quad \tau \to 0^+,$$

has universal local coefficients $a_{2n}(x)$. For the free Dirac fermion in 4D vacuum (no torsion, no gauge field, Lichnerowicz $E = -R/4$), the Christensen-Duff convention gives

$$a_0 = 4\,N_f / (4\pi)^2, \qquad a_2 = -\frac{N_f}{12\,(4\pi)^2}\,R.$$

The $a_4$ density is a polynomial in three curvature scalars with the rationals $-5/(12\cdot 180)$, $+7/(12\cdot 180)$, $-12/(12\cdot 180)$ for $R^2, R_{\mu\nu}^2, R_{\mu\nu\rho\sigma}^2$ respectively (Vassilevich 2003 Eq. 4.40).

In [1]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

import math
from src.core.constants import HEAT_KERNEL_PARAMS, GRAV_PARAMS
from src.heat_kernel import (
    SeeleyDeWittCoefficients,
    seeley_dewitt_coefficients,
    G_N_from_a2,
    a2_calibration_passes,
    a2_calibration_relative_error,
    a4_basis,
    gauss_bonnet_combination,
    higher_curvature_dirac_signs,
)

print(f"Dirac trace dim          tr 1_4 = {HEAT_KERNEL_PARAMS['DIRAC_TRACE_DIM']}")
print(f"a_0 rational                   = {HEAT_KERNEL_PARAMS['A0_DIRAC_RATIONAL']}  (= 4 = tr 1_4)")
print(f"a_2 R-coefficient (per N_f)    = {HEAT_KERNEL_PARAMS['A2_DIRAC_R_COEF']:+.6f} = -1/12")
print(f"(4π)² Gaussian normalization  ≈ {HEAT_KERNEL_PARAMS['FOUR_PI_SQ']:.5f}")
print(f"EH prefactor 12·(4π)²         ≈ {HEAT_KERNEL_PARAMS['EH_PREFACTOR_TWELVE_FOUR_PI_SQ']:.3f}")
print()
print(f"Decision-Gate-E.2 tolerance    = ±{HEAT_KERNEL_PARAMS['A2_GN_MATCH_TOLERANCE']*100:.0f}%")
print(f"GRAV_PARAMS G_N tolerance      = ±{GRAV_PARAMS['G_N_MATCH_TOLERANCE']*100:.0f}%   (same band)")

Dirac trace dim          tr 1_4 = 4
a_0 rational                   = 4.0  (= 4 = tr 1_4)
a_2 R-coefficient (per N_f)    = -0.083333 = -1/12
(4π)² Gaussian normalization  ≈ 157.91367
EH prefactor 12·(4π)²         ≈ 1894.964

Decision-Gate-E.2 tolerance    = ±50%
GRAV_PARAMS G_N tolerance      = ±50%   (same band)


## 2. Leading coefficient $a_0$

The leading term in the expansion. With the Dirac trace dimension absorbed:

$$a_0(N_f) = \frac{4\,N_f}{(4\pi)^2}.$$

Linear in $N_f$, strictly positive for any positive species count. Lean theorems: `a0_dirac_pos`, `a0_dirac_linear`, plus the closed form `a0_dirac` definition.


In [2]:
for N_f in [1, 4, 15, 45, 48]:
    sd = seeley_dewitt_coefficients(N_f)
    closed = 4 * N_f / (4 * math.pi) ** 2
    print(f"N_f = {N_f:>3}   a_0 = {sd.a0:>14.6e}   closed-form = {closed:>14.6e}   match = {math.isclose(sd.a0, closed)}")

N_f =   1   a_0 =   2.533030e-02   closed-form =   2.533030e-02   match = True
N_f =   4   a_0 =   1.013212e-01   closed-form =   1.013212e-01   match = True
N_f =  15   a_0 =   3.799544e-01   closed-form =   3.799544e-01   match = True
N_f =  45   a_0 =   1.139863e+00   closed-form =   1.139863e+00   match = True
N_f =  48   a_0 =   1.215854e+00   closed-form =   1.215854e+00   match = True


## 3. Einstein-Hilbert coefficient $a_2$

The $a_2$ density carries one power of $R$:

$$a_2(N_f, R) = -\frac{N_f}{12\,(4\pi)^2}\,R.$$

Sign: spin-$\tfrac{1}{2}$ Lichnerowicz convention. Strictly negative for positive $N_f$. Lean: `a2_R_coefficient_neg`, `a2_R_coefficient_eq_zero_iff` (vanishing iff $N_f = 0$ — the "no fermion species" trivial case).

In [3]:
for N_f in [1, 15, 45]:
    sd = seeley_dewitt_coefficients(N_f)
    closed = -N_f / (12 * (4 * math.pi) ** 2)
    print(f"N_f = {N_f:>3}   a_2 R-coef = {sd.a2_R_coef:>+14.6e}   closed-form = {closed:>+14.6e}   match = {math.isclose(sd.a2_R_coef, closed)}")
print()
print("Vanishing biconditional check:")
sd0 = seeley_dewitt_coefficients
# a_2 (N_f = 0) is excluded by the dataclass invariant; in Lean
# a2_R_coefficient(0) = 0 directly.
from src.core.formulas import seeley_dewitt_a2_R_coefficient
print(f"  a_2(N_f = 0)    = {seeley_dewitt_a2_R_coefficient(0.0)}    (Lean: a2_R_coefficient_eq_zero_iff)")
print(f"  a_2(N_f = 1e-6) = {seeley_dewitt_a2_R_coefficient(1e-6):+.3e}  (vanishingly small but nonzero)")

N_f =   1   a_2 R-coef =  -5.277145e-04   closed-form =  -5.277145e-04   match = True
N_f =  15   a_2 R-coef =  -7.915717e-03   closed-form =  -7.915717e-03   match = True
N_f =  45   a_2 R-coef =  -2.374715e-02   closed-form =  -2.374715e-02   match = True

Vanishing biconditional check:
  a_2(N_f = 0)    = -0.0    (Lean: a2_R_coefficient_eq_zero_iff)
  a_2(N_f = 1e-6) = -5.277e-10  (vanishingly small but nonzero)


## 4. Tracked-hypothesis structure `DiracHeatKernelAsymptotic`

The PDE-level statement of the heat-kernel asymptotic (Vassilevich 2003 Theorem 4.1) requires manifold + spin-bundle infrastructure not yet in Mathlib. The Lean module encodes the asymptotic *content* as a tracked-hypothesis structure whose invariants `a0_value` and `a2_R_value` force any consumer to commit to the Christensen-Duff coefficient values:

```lean
structure DiracHeatKernelAsymptotic (N_f : ℝ) where
  trace : ℝ → ℝ
  trace_nonneg : ∀ τ, 0 < τ → 0 ≤ trace τ
  a0 : ℝ
  a0_value : a0 = a0_dirac N_f       -- forces textbook value
  a2_R_coef : ℝ
  a2_R_value : a2_R_coef = a2_R_coefficient N_f  -- forces textbook value
  N_f_pos : 0 < N_f
```

Theorems consuming the structure:
- `a0_pos`: structure-side $a_0$ is positive (composes `a0_value` + `a0_dirac_pos`)
- `a2_neg`: structure-side $a_2$ R-coefficient is negative
- `a2_eq_closed_form`: structure-side coefficient equals the textbook value (substantive structure-consuming bridge for downstream calibration)

This is the same pattern as `H_HorizonBoundaryCondition` in Phase 6a.3 `BHEntropyMicroscopic.lean`.

## 5. Decision Gate E.2: $G_N$ from $a_2$ matches Sakharov-Adler

Integrating the $\Lambda^2$-divergent piece of $a_2$ against the volume measure and matching the Einstein-Hilbert coefficient $-1/(16\pi G_N)$ produces

$$G_N \;=\; \frac{12\pi}{N_f \Lambda_{UV}^2}.$$

The Lean theorem `G_N_from_a2_eq_G_N_sakharov` proves this equality against `LinearizedEFE.G_N_sakharov` from Phase 6a.1 — a substantive cross-bridge: the proof body invokes `LinearizedEFE.G_N_sakharov` by name (drift-protection per `feedback_python_lean_refs_drift.md`).

In [4]:
# Closed-form match across the natural Λ_UV grid
print(f"{'Λ_UV (GeV)':>14}  {'N_f':>4}  {'G_N HK (GeV⁻²)':>18}  {'12π/(N_f Λ²)':>18}  match")
for Lambda in [1e10, 1e13, 1e16, 1e19]:
    for N_f in [15, 45]:
        G_hk = G_N_from_a2(Lambda, N_f)
        G_closed = 12 * math.pi / (N_f * Lambda ** 2)
        ok = math.isclose(G_hk, G_closed, rel_tol=1e-12)
        print(f"  {Lambda:>12.2e}  {N_f:>4}  {G_hk:>18.6e}  {G_closed:>18.6e}  {ok}")

    Λ_UV (GeV)   N_f      G_N HK (GeV⁻²)        12π/(N_f Λ²)  match
      1.00e+10    15        2.513274e-20        2.513274e-20  True
      1.00e+10    45        8.377580e-21        8.377580e-21  True
      1.00e+13    15        2.513274e-26        2.513274e-26  True
      1.00e+13    45        8.377580e-27        8.377580e-27  True
      1.00e+16    15        2.513274e-32        2.513274e-32  True
      1.00e+16    45        8.377580e-33        8.377580e-33  True
      1.00e+19    15        2.513274e-38        2.513274e-38  True
      1.00e+19    45        8.377580e-39        8.377580e-39  True


## 6. Decision Gate E.2 biconditional

The load-bearing correctness-push punchline:

> Heat-kernel $G_N(\Lambda, N_f) = G_N^{\mathrm{emerg}}(\Lambda, N_f, \alpha_{\mathrm{ADW}})$ at fixed $(\Lambda, N_f > 0)$ **iff** $\alpha_{\mathrm{ADW}} = 1$.

(Lean: `a2_matches_GNemerg_iff_alpha_ADW_unity`.)

Forward direction uses `mul_right_cancel₀` against `LinearizedEFE.G_N_sakharov_pos`. Reverse invokes `LinearizedEFE.G_N_emerg_at_alpha_one`. The biconditional is the formal sense in which the heat-kernel calibration pins $\alpha_{\mathrm{ADW}}$: any future Vergeles-unitarity computation returning $\alpha_{\mathrm{ADW}} \ne 1$ is detectable as a deviation from the textbook Christensen-Duff $-1/12$ coefficient.


In [5]:
# Tabulate the relative error around α_ADW = 1
print(f"{'α_ADW':>8}  {'rel err':>10}  {'pass band ±50%':>16}")
for alpha in [0.1, 0.5, 0.667, 0.9, 1.0, 1.1, 1.5, 2.0, 5.0]:
    err = a2_calibration_relative_error(1e16, 15, alpha)
    pas = a2_calibration_passes(1e16, 15, alpha)
    print(f"  {alpha:>6.3f}  {err:>10.4f}  {('PASS' if pas else 'FAIL'):>16}")
print()
print("Boundary: α = 2  →  rel err = 0.5 (tolerance band edge); ≤ → True")
print("Lean: a2_matches_GNemerg_iff_alpha_ADW_unity (biconditional, no tolerance)")

   α_ADW     rel err    pass band ±50%
   0.100      9.0000              FAIL
   0.500      1.0000              FAIL
   0.667      0.4993              PASS
   0.900      0.1111              PASS
   1.000      0.0000              PASS
   1.100      0.0909              PASS
   1.500      0.3333              PASS
   2.000      0.5000              PASS
   5.000      0.8000              FAIL

Boundary: α = 2  →  rel err = 0.5 (tolerance band edge); ≤ → True
Lean: a2_matches_GNemerg_iff_alpha_ADW_unity (biconditional, no tolerance)


## 7. Higher-curvature $a_4$ basis + Gauss-Bonnet algebra

The $a_4$ density splits across three scalars $\{R^2, R_{\mu\nu}R^{\mu\nu}, R_{\mu\nu\rho\sigma}R^{\mu\nu\rho\sigma}\}$ with rational coefficients
$\{-5/(12\cdot 180),\,+7/(12\cdot 180),\,-12/(12\cdot 180)\}$ per fermion species, modulo the universal $1/(4\pi)^2$ Gaussian normalization.

The Gauss-Bonnet density $\mathcal{G} = R^2 - 4 R_{\mu\nu}^2 + R_{\mu\nu\rho\sigma}^2$ is topological in 4D — its integral on a closed manifold is $32\pi^2 \chi(M)$. At the local-algebra level, the Dirac $a_4$ contribution to $\mathcal{G}$ evaluates to $-N_f/(48\,(4\pi)^2)$ (Lean: `a4_gauss_bonnet_combination` proved by `ring`).

In [6]:
for N_f in [1, 15, 45]:
    a4 = a4_basis(N_f)
    signs = higher_curvature_dirac_signs(N_f)
    gb = gauss_bonnet_combination(N_f)
    closed_gb = -N_f / (48.0 * (4.0 * math.pi) ** 2)
    print(f"N_f = {N_f:>3}  a4 R²: {a4['R_sq']:+.4e} ({signs['R_sq']})   "
          f"Ricci²: {a4['Ricci_sq']:+.4e} ({signs['Ricci_sq']})   "
          f"Riem²: {a4['Riemann_sq']:+.4e} ({signs['Riemann_sq']})")
    print(f"      GB combination: {gb:+.4e}   closed-form -N_f/(48 (4π)²): {closed_gb:+.4e}   match = {math.isclose(gb, closed_gb)}")

N_f =   1  a4 R²: -1.4659e-05 (neg)   Ricci²: +2.0522e-05 (pos)   Riem²: -3.5181e-05 (neg)
      GB combination: -1.3193e-04   closed-form -N_f/(48 (4π)²): -1.3193e-04   match = True
N_f =  15  a4 R²: -2.1988e-04 (neg)   Ricci²: +3.0783e-04 (pos)   Riem²: -5.2771e-04 (neg)
      GB combination: -1.9789e-03   closed-form -N_f/(48 (4π)²): -1.9789e-03   match = True
N_f =  45  a4 R²: -6.5964e-04 (neg)   Ricci²: +9.2350e-04 (pos)   Riem²: -1.5831e-03 (neg)
      GB combination: -5.9368e-03   closed-form -N_f/(48 (4π)²): -5.9368e-03   match = True


## 8. Figure: $a_2$ calibration scan + $G_N(\Lambda_{UV})$ vs CODATA

Two-panel diagnostic for the Decision Gate E.2 anchor:
- **Left:** relative error $|G_N^{(a_2)} - G_N^{\mathrm{emerg}}|/G_N^{\mathrm{emerg}}$ over $\alpha_{\mathrm{ADW}} \in [0.05, 5.0]$ at the GUT-anchor parameters $(\Lambda_{UV}, N_f) = (10^{16}\,\mathrm{GeV}, 15)$. Exact zero at $\alpha_{\mathrm{ADW}} = 1$. The $\pm 50\%$ pass band (matching `GRAV_PARAMS.G_N_MATCH_TOLERANCE`) is shaded.
- **Right:** log–log $G_N(\Lambda_{UV})$ at fixed $N_f = 15$ over $[10^{10}, 10^{19}]\,\mathrm{GeV}$ vs the CODATA reference $G_N^{\mathrm{obs}} = 6.71 \times 10^{-39}\,\mathrm{GeV}^{-2}$. The heat-kernel curve crosses CODATA near $M_P = 1.22 \times 10^{19}\,\mathrm{GeV}$, confirming the Sakharov-Adler scale anchor.

Lean: `G_N_from_a2_eq_G_N_sakharov`, `a2_matches_GNemerg_iff_alpha_ADW_unity`, `G_N_from_a2_at_GUT_inverse`, `G_N_from_a2_inverse_at_GUT_below_planck_squared`.

In [7]:
# viz-ref: fig_a2_vs_linearized_G_N
from src.core.visualizations import fig_a2_vs_linearized_G_N
fig = fig_a2_vs_linearized_G_N()
fig.show()

## 9. Lean theorem inventory (`HeatKernelExpansion.lean`, 19 substantive theorems)

| § | Theorem | What it proves |
|---|---------|----------------|
| 1 | `fourPiSq_pos` | $(4\pi)^2 > 0$ |
| 1 | `fourPiSqInv_pos` | $1/(4\pi)^2 > 0$ |
| 2 | `a0_dirac_pos` | $a_0(N_f) > 0$ for $N_f > 0$ |
| 2 | `a0_dirac_linear` | $a_0(k\,N_f) = k\,a_0(N_f)$ |
| 3 | `a2_R_coefficient_neg` | $a_2$ R-coefficient is negative for $N_f > 0$ |
| 3 | `a2_R_coefficient_linear` | $a_2$ is linear in $N_f$ |
| 3 | `a2_R_coefficient_eq_zero_iff` | $a_2 = 0 \iff N_f = 0$ |
| 4 | `DiracHeatKernelAsymptotic.a0_pos` | structure-side $a_0$ is positive |
| 4 | `DiracHeatKernelAsymptotic.a2_neg` | structure-side $a_2$ R-coef is negative |
| 4 | `DiracHeatKernelAsymptotic.a2_eq_closed_form` | structure-side coef = textbook value |
| 5 | `G_N_from_a2_pos` | $G_N$ is positive for positive $\Lambda, N_f$ |
| 5 | `G_N_from_a2_eq_G_N_sakharov` | **load-bearing cross-bridge** to 6a.1 |
| 6 | `G_N_from_a2_at_GUT_inverse` | $1/G_N$ at GUT anchor = $15 \cdot 10^{32} / (12\pi)$ |
| 6 | `G_N_from_a2_inverse_at_GUT_below_planck_squared` | $1/G_N(\mathrm{GUT}) < M_P^2$ via `norm_num` + `pi_gt_three` |
| 7 | `a4_R_sq_coef_neg` | $R^2$ coefficient is negative |
| 7 | `a4_Ricci_sq_coef_pos` | $R_{\mu\nu}^2$ coefficient is positive |
| 7 | `a4_Riemann_sq_coef_neg` | $R_{\mu\nu\rho\sigma}^2$ coefficient is negative |
| 7 | `a4_gauss_bonnet_combination` | $c_{R^2} - 4c_{R_{\mu\nu}^2} + c_{R_{\mu\nu\rho\sigma}^2} = -N_f/(48(4\pi)^2)$ |
| 8 | `a2_matches_GNemerg_iff_alpha_ADW_unity` | **Decision Gate E.2 biconditional** |

Discipline metric: 6e.1 = 2 retroactive theorems (post-wave audit deleted unused identity wrapper, restructured a defining-the-conclusion P5, and replaced an `rfl`-tautology numerical anchor with a substantive `one_div_div`-using inverse anchor).
